# Phase 3 : Deep Learning - CNN (PyTorch) et MLP (Scikit-Learn)
Dans ce notebook, nous combinons une architecture de pointe en PyTorch (Réseau de Neurones Convolutif 1D) avec une approche Deep Learning plus classique (Multilayer Perceptron) afin de comparer leurs performances sur du texte pré-traité.

In [9]:
import pandas as pd
import numpy as np
import os
import sys
import pickle
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.neural_network import MLPClassifier
from collections import Counter

sys.path.append(os.path.abspath('..'))
from src.data_loader import load_yelp_sample
from src.preprocessing import clean_text

## PARTIE 1 : Réseau de Neurones Convolutif 1D (PyTorch)
### 1. Préparation des données (Tokenization et Plongements)

In [10]:
# 1. Chargement et Nettoyage
print("Chargement des données...")
df = load_yelp_sample('../data/raw/review.json', n_rows=50000)
df['label'] = (df['stars'] <= 2).astype(int)

print("Nettoyage du texte en cours...")
df['text'] = df['text'].apply(clean_text)

# 2. Tokenization basique
texts = df['text'].str.split().tolist()
labels = df['label'].tolist()

# 3. Création du vocabulaire
vocab_size = 10000
all_words = [word for text in texts for word in text]
word_counts = Counter(all_words)
vocab = {word: i+2 for i, (word, _) in enumerate(word_counts.most_common(vocab_size - 2))}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1

# 4. Conversion et Padding
max_len = 100
X_seq = []
for text in texts:
    seq = [vocab.get(word, vocab['<UNK>']) for word in text]
    if len(seq) < max_len:
        seq = seq + [vocab['<PAD>']] * (max_len - len(seq))
    else:
        seq = seq[:max_len]
    X_seq.append(seq)

# 5. Tenseurs et Séparation
X_tensor = torch.tensor(X_seq, dtype=torch.long)
y_tensor = torch.tensor(labels, dtype=torch.float32)
X_train, X_test, y_train, y_test_cnn = train_test_split(X_tensor, y_tensor, test_size=0.2, random_state=42)

# 6. DataLoaders
batch_size = 64
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test_cnn), batch_size=batch_size, shuffle=False)
print(f"Taille de l'entraînement : {len(X_train)} avis. Taille du test : {len(X_test)} avis.")

Chargement des données...
Chargement de 50000 lignes depuis ../data/raw/review.json...
Nettoyage du texte en cours...
Taille de l'entraînement : 40000 avis. Taille du test : 10000 avis.


### 2. Architecture du Modèle PyTorch (TextCNN)

In [11]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_filters, filter_sizes):
        super(TextCNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embed_dim, out_channels=num_filters, kernel_size=fs)
            for fs in filter_sizes
        ])
        self.fc = nn.Linear(len(filter_sizes) * num_filters, 1)
        self.dropout = nn.Dropout(0.5)
        self.sigmoid = nn.Sigmoid()

    def forward(self, text):
        embedded = self.embedding(text)
        embedded = embedded.permute(0, 2, 1)
        conved = [torch.relu(conv(embedded)) for conv in self.convs]
        pooled = [torch.max(conv, dim=2)[0] for conv in conved]
        cat = self.dropout(torch.cat(pooled, dim=1))
        return self.sigmoid(self.fc(cat)).squeeze()

### 3. Entraînement du Modèle

In [ ]:
embed_dim = 100
num_filters = 100
filter_sizes = [2, 3, 4]
num_epochs = 5
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = TextCNN(vocab_size, embed_dim, num_filters, filter_sizes).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Début de l'entraînement du CNN...") 
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{num_epochs} | Loss moyenne : {total_loss/len(train_loader):.4f}")

Début de l'entraînement du CNN...
Epoch 1/15 | Loss moyenne : 0.3852
Epoch 2/15 | Loss moyenne : 0.2767
Epoch 3/15 | Loss moyenne : 0.2345
Epoch 4/15 | Loss moyenne : 0.2081
Epoch 5/15 | Loss moyenne : 0.1813
Epoch 6/15 | Loss moyenne : 0.1602
Epoch 7/15 | Loss moyenne : 0.1397
Epoch 8/15 | Loss moyenne : 0.1203
Epoch 9/15 | Loss moyenne : 0.0998
Epoch 10/15 | Loss moyenne : 0.0846
Epoch 11/15 | Loss moyenne : 0.0717
Epoch 12/15 | Loss moyenne : 0.0587
Epoch 13/15 | Loss moyenne : 0.0499
Epoch 14/15 | Loss moyenne : 0.0435
Epoch 15/15 | Loss moyenne : 0.0386


### 4. Évaluation des Performances du CNN (PyTorch)

In [13]:
model.eval() # On passe en mode évaluation (désactive le Dropout)
y_pred_cnn = []
y_true_cnn = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        predictions = model(batch_X)
        # On arrondit la probabilité (> 0.5 devient 1, sinon 0)
        predicted_classes = (predictions >= 0.5).float()
        y_pred_cnn.extend(predicted_classes.cpu().numpy())
        y_true_cnn.extend(batch_y.cpu().numpy())

print("Rapport de Classification (TextCNN PyTorch) :")
print(classification_report(y_true_cnn, y_pred_cnn))

Rapport de Classification (TextCNN PyTorch) :
              precision    recall  f1-score   support

         0.0       0.94      0.94      0.94      7710
         1.0       0.80      0.78      0.79      2290

    accuracy                           0.91     10000
   macro avg       0.87      0.86      0.87     10000
weighted avg       0.91      0.91      0.91     10000



---
## PARTIE 2 : Les Modèles MLP (Scikit-Learn)
On récupère les données nettoyées et on les vectorise avec TF-IDF pour comparer avec l'approche Scikit-Learn.

In [14]:
print("Chargement du vectorizer généré dans le Notebook 2...")
vec = pickle.load(open('../models/vectorizer.pkl', 'rb'))

# Utilisation du df dejà nettoyé en haut de ce notebook
X_mlp = vec.transform(df['text'])
y_mlp = df['label']

Chargement du vectorizer généré dans le Notebook 2...


### Architecture 1 : Standard (64, 32)

In [15]:
mlp1 = MLPClassifier(hidden_layer_sizes=(64, 32), early_stopping=True, random_state=42)
mlp1.fit(X_mlp, y_mlp)
report_m1 = classification_report(y_mlp, mlp1.predict(X_mlp), output_dict=True)
display(pd.DataFrame(report_m1).transpose().round(2))

,precision,recall,f1-score,support
0,0.95,0.97,0.96,38469.00
1,0.88,0.85,0.86,11531.00
accuracy,0.94,0.94,0.94,0.94
macro avg,0.92,0.91,0.91,50000.00
weighted avg,0.94,0.94,0.94,50000.00


### Architecture 2 : Profonde (128, 64, 32)

In [16]:
mlp2 = MLPClassifier(hidden_layer_sizes=(128, 64, 32), early_stopping=True, random_state=42)
mlp2.fit(X_mlp, y_mlp)
report_m2 = classification_report(y_mlp, mlp2.predict(X_mlp), output_dict=True)
display(pd.DataFrame(report_m2).transpose().round(2))

,precision,recall,f1-score,support
0,0.96,0.96,0.96,38469.00
1,0.87,0.87,0.87,11531.00
accuracy,0.94,0.94,0.94,0.94
macro avg,0.92,0.92,0.92,50000.00
weighted avg,0.94,0.94,0.94,50000.00


### Architecture 3 : Large (256)

In [17]:
mlp3 = MLPClassifier(hidden_layer_sizes=(256,), early_stopping=True, random_state=42)
mlp3.fit(X_mlp, y_mlp)
report_m3 = classification_report(y_mlp, mlp3.predict(X_mlp), output_dict=True)
display(pd.DataFrame(report_m3).transpose().round(2))

pickle.dump(mlp1, open('../models/deep_model.pkl', 'wb'))
print("Modèle MLP sauvegardé.")

,precision,recall,f1-score,support
0,0.96,0.96,0.96,38469.00
1,0.87,0.85,0.86,11531.00
accuracy,0.94,0.94,0.94,0.94
macro avg,0.91,0.91,0.91,50000.00
weighted avg,0.94,0.94,0.94,50000.00


Modèle MLP sauvegardé.
